# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [ ]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [ ]:
# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    '''Calculate math expressions.'''
    try:
        result = eval(expression)  # Safe in controlled environment
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}

# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    '''Get current time.'''
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }

print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '15:36:04', 'date': '2026-09-19', 'day': 'Saturday'}


## Part 2: Tool Registry

Manage available tools.

In [ ]:
class ToolRegistry:
    '''Manages available tools for the agent.'''

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)

#Create registry
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')

print(registry.list_tools())

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [ ]:
class SimpleReActAgent:
    '''Simple agent that can use tools.'''

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take.'''
        # Simplified: pattern matching for demo
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            # Extract math expression from query
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        # Step 1: Thought
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        # Step 2: Action
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Step 3: Answer
        if 'result' in observation:
            answer = f"The result is {observation['result']}"
        elif 'time' in observation:
            answer = f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'error' in observation:
            answer = f"Error: {observation['error']}"
        else:
            answer = "I can help with calculations and time queries."

        return answer

# Test the agent
agent = SimpleReActAgent(registry)

print('\nAgent Tests:')
print('Q: What is 25 * 4?')
print('A:', agent.run('What is 25 * 4?'))
print('\nQ: What time is it?')
print('A:', agent.run('What time is it?'))


Agent Tests:
Q: What is 25 * 4?
A: Error: invalid syntax (<string>, line 0)

Q: What time is it?
A: It's 15:38:58 on 2026-09-19 (Saturday)


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [ ]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 15:39:28 on 2026-09-19 (Saturday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [1]:
from typing import Dict, Any, List, Optional
import json

# Simulated in-memory storage for our digital twin tools
HABIT_LOG = {
    "hydration": "Logged 2.5 Liters of water today.",
    "exercise": "Completed 45 minutes of strength training.",
    "reading": "Read 20 pages on systems thinking and engineering."
}

TASK_LIST = [
    {"task": "Finalize UMaT mechanical engineering thesis formatting", "status": "In Progress"},
    {"task": "Review Uriel Constellation youth leadership agenda", "status": "Pending"},
    {"task": "Test solar-powered irrigation prototype", "status": "Completed"}
]

# ==========================================
# 1. Implement Custom Tools
# ==========================================

def habit_tracker(habit_name: str) -> Dict[str, Any]:
    """
    Retrieves the current status or log of a specific daily habit.

    Args:
        habit_name (str): The name of the habit to check (e.g., 'hydration', 'exercise', 'reading').

    Returns:
        dict: The result containing the habit status or a message if not found.
    """
    key = habit_name.strip().lower()
    if key in HABIT_LOG:
        return {"result": HABIT_LOG[key]}
    return {"result": f"No habit log found for '{habit_name}'."}


def task_manager(action: str, task_name: Optional[str] = None) -> Dict[str, Any]:
    """
    Manages the digital twin's task list by viewing current tasks or adding a new task.

    Args:
        action (str): Either 'view' to list tasks or 'add' to insert a new task.
        task_name (str, optional): The description of the task when action is 'add'.

    Returns:
        dict: The updated task list or status confirmation.
    """
    action = action.strip().lower()
    if action == "view":
        return {"result": TASK_LIST}
    elif action == "add":
        if not task_name:
            return {"result": "ERROR: 'task_name' must be provided when adding a task."}
        new_task = {"task": task_name.strip(), "status": "Pending"}
        TASK_LIST.append(new_task)
        return {"result": f"Successfully added task: '{task_name}'", "tasks": TASK_LIST}
    else:
        return {"result": f"ERROR: Unknown action '{action}'. Use 'view' or 'add'."}


# ==========================================
# 2. Mocking Digital Twin Tool Integration
# ==========================================
class MockDigitalTwin:
    """Mock Digital Twin class demonstrating tool registration and dispatch."""
    def __init__(self):
        self.tool_registry = {}

    def add_tool(self, name: str, func: callable, description: str):
        self.tool_registry[name] = {
            "function": func,
            "description": description
        }
        print(f"Registered tool: '{name}' -> {description}")

    def chat(self, query: str) -> str:
        print(f"\n[User Query]: {query}")
        # In a full ReAct loop, the LLM decides which tool to call based on the query.
        # Here we demonstrate manual dispatch matching the query intent:
        if "habit" in query.lower() or "hydration" in query.lower():
            res = self.tool_registry["habit_tracker"]["function"]("hydration")
            return f"Tool Output: {res['result']}"
        elif "task" in query.lower():
            res = self.tool_registry["task_manager"]["function"]("view")
            return f"Tool Output: {res['result']}"
        return "Processed query without tool invocation."


# ==========================================
# 3. Test Your Implementation
# ==========================================
if __name__ == "__main__":
    twin = MockDigitalTwin()

    # Register custom tools
    twin.add_tool(
        "habit_tracker",
        habit_tracker,
        "Retrieves daily habit logs and wellness progress."
    )
    twin.add_tool(
        "task_manager",
        task_manager,
        "Views current tasks or adds new tasks to the productivity tracker."
    )

    # Test cases
    print(twin.chat("Can you check my hydration habit status?"))
    print(twin.chat("Show me my current task list."))

    # Direct function tests
    print("\nDirect habit tracker test:", habit_tracker("exercise"))
    print("Direct task manager add test:", task_manager("add", "Prepare presentation slides for ISOC community"))

Registered tool: 'habit_tracker' -> Retrieves daily habit logs and wellness progress.
Registered tool: 'task_manager' -> Views current tasks or adds new tasks to the productivity tracker.

[User Query]: Can you check my hydration habit status?
Tool Output: Logged 2.5 Liters of water today.

[User Query]: Show me my current task list.
Tool Output: [{'task': 'Finalize UMaT mechanical engineering thesis formatting', 'status': 'In Progress'}, {'task': 'Review Uriel Constellation youth leadership agenda', 'status': 'Pending'}, {'task': 'Test solar-powered irrigation prototype', 'status': 'Completed'}]

Direct habit tracker test: {'result': 'Completed 45 minutes of strength training.'}
Direct task manager add test: {'result': "Successfully added task: 'Prepare presentation slides for ISOC community'", 'tasks': [{'task': 'Finalize UMaT mechanical engineering thesis formatting', 'status': 'In Progress'}, {'task': 'Review Uriel Constellation youth leadership agenda', 'status': 'Pending'}, {'tas

## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI